# Which parts of CPython work in a browser

This project promises that the first experiments in every lesson run in a browser tab with nothing installed. That rests on Pyodide, which is CPython compiled to WebAssembly, and on the introspection surfaces the lessons poke at surviving that build. Some of them do not, and the point of this notebook is to find out which, on the exact runtime you are sitting in front of rather than on one we tested a while ago.

Run every cell. It takes a few seconds and installs nothing. At the end you get a table of what worked here, and you can compare it against the recordings committed next to this file.

One warning worth reading first. One of the checks below asks what happens when the bytecode optimizer is handed a constants list that is too short. On a normal build that raises a tidy exception. In a WebAssembly build it reads past the end of memory and takes the whole runtime with it, which in a notebook means the kernel dies and you have to restart it. That check is last for exactly this reason, and everything above it will have already printed.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/probes/pyodide/probe.ipynb)

## The checks

Each one is a small piece of Python that leaves its answer in a variable called `result`. They are written out in full rather than imported, so you can read what is being asked, and edit one to ask something else.

In [ ]:
CHECKS = {
    # Which CPython is this, and what was it built for
    "version": """
import platform
import sys
import sysconfig

result = {
    "python": platform.python_version(),
    "platform": sysconfig.get_platform(),
    "pointer_bytes": sys.maxsize.bit_length() // 8 + 1,
    "free_threaded": bool(sysconfig.get_config_var("Py_GIL_DISABLED")),
}
""",
    # Is _testinternalcapi importable at all
    "internal_capi_import": """
import _testinternalcapi

wanted = ("compiler_codegen", "optimize_cfg", "assemble_code_object")
result = {name: hasattr(_testinternalcapi, name) for name in wanted}
""",
    # Does compiler_codegen turn a tree into an instruction sequence
    "compiler_codegen": """
import _testinternalcapi
import ast

sequence, metadata = _testinternalcapi.compiler_codegen(ast.parse("answer = 6 * 7"), "<probe>", 0)
instructions = sequence.get_instructions()
result = {
    "instructions": len(instructions),
    "first": instructions[0][0],
    "metadata_keys": sorted(metadata),
}
""",
    # Does optimize_cfg run over that sequence the way pyxray calls it
    "optimize_cfg": """
import _testinternalcapi
import ast

sequence, metadata = _testinternalcapi.compiler_codegen(ast.parse("answer = 6 * 7"), "<probe>", 0)
optimized = _testinternalcapi.optimize_cfg(sequence, metadata["consts"], 0)
result = {"instructions": len(optimized.get_instructions())}
""",
    # Does optimize_cfg run at all, given a constants list built by hand
    "optimize_cfg_direct": """
import _testinternalcapi
import ast
import opcode

sequence, metadata = _testinternalcapi.compiler_codegen(ast.parse("answer = 6 * 7"), "<probe>", 0)
load = opcode.opmap["LOAD_CONST"]
slots = [one[1] for one in sequence.get_instructions() if one[0] == load]
consts = [None] * (max(slots) + 1 if slots else 0)
optimized = _testinternalcapi.optimize_cfg(sequence, consts, 0)
result = {"slots": len(consts), "instructions": len(optimized.get_instructions())}
""",
    # Can ctypes read the two fields in front of every object
    "ctypes_header": """
import ctypes
import sys

value = [1, 2, 3]
word = ctypes.sizeof(ctypes.c_ssize_t)
result = {
    "refcount_field": ctypes.c_ssize_t.from_address(id(value)).value,
    "getrefcount": sys.getrefcount(value),
    "type_pointer_matches": ctypes.c_void_p.from_address(id(value) + word).value == id(list),
    "word_bytes": word,
}
""",
    # Does sys.monitoring register a callback and fire it
    "monitoring": """
import sys

TOOL = 5
seen = []
sys.monitoring.use_tool_id(TOOL, "wasmprobe")
try:
    def target():
        return 1 + 1
    sys.monitoring.register_callback(
        TOOL, sys.monitoring.events.PY_START, lambda *arguments: seen.append(arguments)
    )
    sys.monitoring.set_local_events(TOOL, target.__code__, sys.monitoring.events.PY_START)
    target()
finally:
    sys.monitoring.free_tool_id(TOOL)
allowed = [
    name
    for name in dir(sys.monitoring.events)
    if name.isupper() and not name.startswith("NO_")
]
result = {"fired": len(seen), "event_names": len(allowed)}
""",
    # Does sys.settrace still see call, line and return
    "settrace": """
import sys

seen = []
def tracer(frame, event, argument):
    seen.append(event)
    return tracer
def target():
    return 2
sys.settrace(tracer)
try:
    target()
finally:
    sys.settrace(None)
result = {"events": seen}
""",
    # Does the cycle collector behave the way T09 says it does
    "gc": """
import gc
import weakref

class Node:
    pass
first, second = Node(), Node()
watch = weakref.ref(first)
first.other, second.other = second, first
del first, second
gc.collect()
# What gc.collect() returns counts everything it swept, including whatever else this
# process happened to be holding, so it is different every time. Whether this particular
# cycle went away is the question the lesson actually asks.
result = {
    "cycle_freed": watch() is None,
    "thresholds": list(gc.get_threshold()),
    "enabled": gc.isenabled(),
    "generations": len(gc.get_stats()),
}
""",
    # Does sys._debugmallocstats produce anything under Emscripten's allocator
    "debugmallocstats": """
import sys

result = {"callable": callable(getattr(sys, "_debugmallocstats", None))}
""",
    # Do dis, ast, symtable, tokenize and marshal all import
    "front_end_modules": """
found = {}
for name in ("dis", "ast", "symtable", "tokenize", "marshal", "opcode", "_opcode"):
    try:
        __import__(name)
        found[name] = True
    except Exception as error:
        found[name] = f"{type(error).__name__}: {error}"
result = found
""",
    # Does dis give the same instructions as a native interpreter
    "disassembly": """
import dis
import marshal

source = "answer = 6 * 7"
code = compile(source, "<probe>", "exec")
result = {
    "opnames": [one.opname for one in dis.get_instructions(source)],
    "code_size": len(code.co_code),
    "consts": [repr(one) for one in code.co_consts],
    "marshal_size": len(marshal.dumps(code)),
}
""",
    # Where does the shared range of small integers stop
    "small_integers": """
top = 0
for candidate in range(0, 4096):
    if int(str(candidate)) is int(str(candidate)):
        top = candidate
result = {"top": top}
""",
    # Can a thread be started
    "threading": """
import threading

ran = []
worker = threading.Thread(target=lambda: ran.append(True))
worker.start()
worker.join()
result = {"ran": ran == [True], "active": threading.active_count()}
""",
}

print(f"{len(CHECKS)} checks")

## Running them

Each check gets a fresh namespace and anything it throws is caught and recorded, so one failure does not stop the rest. A check that takes the runtime down cannot be caught, which is why the dangerous one is separated out below.

In [ ]:
answers = {}
for key, source in CHECKS.items():
    namespace = {}
    try:
        exec(source, namespace)
    except BaseException as error:
        answers[key] = ("raised", f"{type(error).__name__}: {error}")
    else:
        answers[key] = ("ok", namespace.get("result"))

print(f"{sum(1 for status, _ in answers.values() if status == 'ok')} of {len(answers)} worked")

### The table

The same shape as the matrix in `report.md` next to this notebook, so the two are easy to read against each other. The answer column is whatever the check left in `result`.

In [ ]:
def matrix(answers):
    """Print what happened, one row per check."""
    width = max(len(key) for key in answers)
    print(f"{'check'.ljust(width)}  status  answer")
    print("-" * (width + 40))
    for key, (status, answer) in answers.items():
        print(f"{key.ljust(width)}  {status.ljust(6)}  {answer}")


matrix(answers)

## The one that can kill the kernel

Everything above has already printed, so run this last. The cell catches the exception itself, so on a normal build you get a tidy `ValueError` naming the constant it could not find. In a browser there is nothing to catch: the read goes past the end of memory, the runtime does not come back, and you restart the kernel.

That difference is the reason the pipeline widget in the lessons builds its own constants list rather than trusting the one it is handed.

In [ ]:
import _testinternalcapi
import ast

sequence, metadata = _testinternalcapi.compiler_codegen(ast.parse("answer = 6 * 7"), "<probe>", 0)
try:
    _testinternalcapi.optimize_cfg(sequence, [], 0)
    result = {"raised": None}
except BaseException as error:
    result = {"raised": f"{type(error).__name__}: {error}"}

If the cell above came back rather than killing the kernel, here is what it caught. If the kernel died, that is the answer, and it is the one worth telling us about.

In [ ]:
print(result)

## What to do with this

Compare what you got against `report.md` in this directory, which is the same matrix recorded on a native CPython and on Pyodide under Node.

A check that failed here and passed there is usually your environment: an old Pyodide, a sandbox that blocks threads, a Python built without the test modules. A check that failed in both is our problem, and worth an issue.